# 1. Data Loading and Preparation
In this section, the benchmark cases and chatbot guidelines are loaded and prepared for further use in the pipeline.

In [ ]:
%pip install -q -U "transformers==4.45.0" "accelerate" "bitsandbytes>=0.46.1" "sentencepiece"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 62.9 MB/s eta 0:00:00


In [ ]:
%pip install -q "lingua-language-detector"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.3/170.3 MB 7.2 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import pathlib
import pandas as pd
import json
import time
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import torch
import transformers
import accelerate
import bitsandbytes as bnb
import time
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM
from transformers import GenerationConfig
from lingua import Language, LanguageDetectorBuilder

In [ ]:
# file paths
DATA_DIR = Path("/content")

IN_SCOPE_CASES_FILE = DATA_DIR / "in_scope_QA.csv"
OUT_OF_SCOPE_CASES_FILE = DATA_DIR / "out_of_scope_QA.csv"
GUIDELINES_FILE = DATA_DIR / "QA_guidelines.csv"

In [ ]:
in_scope_cases_df = pd.read_csv(IN_SCOPE_CASES_FILE)
out_of_scope_cases_df = pd.read_csv(OUT_OF_SCOPE_CASES_FILE)
guidelines_df = pd.read_csv(GUIDELINES_FILE)

print("In-scope cases shape:", in_scope_cases_df.shape)
print("Out-of-scope cases shape:", out_of_scope_cases_df.shape)
print("Guidelines shape:", guidelines_df.shape)

print("\nIn-scope columns:")
print(in_scope_cases_df.columns.tolist())

print("\nOut-of-scope columns:")
print(out_of_scope_cases_df.columns.tolist())

print("\nGuidelines columns:")
print(guidelines_df.columns.tolist())

In-scope cases shape: (50, 9)
Out-of-scope cases shape: (30, 9)
Guidelines shape: (63, 2)

In-scope columns:
['case_id', 'question_en', 'question_nl', 'context', 'reference_answer_en', 'reference_answer_nl', 'expected_answer_points', 'should_not_include', 'source_basis']

Out-of-scope columns:
['case_id', 'question_en', 'question_nl', 'context', 'reference_answer_en', 'reference_answer_nl', 'expected_answer_points', 'should_not_include', 'source_basis']

Guidelines columns:
['guideline_type', 'guideline_text']


In [ ]:
required_case_columns = [
    "case_id",
    "question_en",
    "question_nl",
    "context",
    "reference_answer_en",
    "reference_answer_nl",
    "expected_answer_points",
    "should_not_include",
    "source_basis",
]

required_guideline_columns = [
    "guideline_type",
    "guideline_text",
]

In [1]:
def validate_case_dataframe(df: pd.DataFrame, dataframe_name: str) -> None:
    """
    Check whether a case dataframe contains all required columns.
    """
    missing_columns = [col for col in required_case_columns if col not in df.columns]

    if missing_columns:
        raise ValueError(f"{dataframe_name} is missing required columns: {missing_columns}")

NameError: name 'pd' is not defined

In [2]:
# clean text fields
def clean_case_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean text fields in a case dataframe.
    """

    cleaned_df = df.copy()

    for col in required_case_columns:
        cleaned_df[col] = cleaned_df[col].fillna("").astype(str).str.strip()

    return cleaned_df

NameError: name 'pd' is not defined

In [3]:
def validate_guidelines_dataframe(df: pd.DataFrame) -> None:
    """
    Check whether the guidelines dataframe contains all required columns.
    """
    missing_columns = [col for col in required_guideline_columns if col not in df.columns]

    if missing_columns:
        raise ValueError(f"Guidelines dataframe is missing required columns: {missing_columns}")

NameError: name 'pd' is not defined

In [4]:
def clean_guidelines_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean text fields in the guidelines dataframe.
    """

    cleaned_df = df.copy()

    cleaned_df["guideline_type"] = (
        cleaned_df["guideline_type"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    cleaned_df["guideline_text"] = (
        cleaned_df["guideline_text"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    return cleaned_df

NameError: name 'pd' is not defined

In [5]:
# schema checks
validate_case_dataframe(in_scope_cases_df, "in_scope_cases_df")
validate_case_dataframe(out_of_scope_cases_df, "out_of_scope_cases_df")
validate_guidelines_dataframe(guidelines_df)

in_scope_cases_df = clean_case_dataframe(in_scope_cases_df)
out_of_scope_cases_df = clean_case_dataframe(out_of_scope_cases_df)
guidelines_df = clean_guidelines_dataframe(guidelines_df)


print("All files validated and cleaned.")

NameError: name 'validate_case_dataframe' is not defined

In [ ]:
guideline_dict = (
    guidelines_df
    .groupby("guideline_type")["guideline_text"]
    .apply(list)
    .to_dict()
)

print("Guideline types:")
print(list(guideline_dict.keys()))

print("\nNumber of guidelines:", len(guidelines_df))

Guideline types:
['Answers should not include:', 'For EPV, policy, cost, and billing-related questions, the chatbot should:', 'For all answers, the chatbot should:', 'For anomaly follow-up and recommendation questions, a good answer should:', 'For answer format, the chatbot should:', 'For app/data interpretation questions, a good answer should:', 'For explanation questions, a good answer should:', 'For grounding and evidence, the chatbot should:', 'For language behaviour, the chatbot should:', 'For model identity, the chatbot should:', 'For privacy and personalisation, the chatbot should:', 'For technical malfunction or missing-data questions, the chatbot should:', 'For unrelated or out-of-scope questions, the chatbot should:', 'When evidence is incomplete, the chatbot should:']

Number of guidelines: 63


In [ ]:
RETRIEVAL_TEST_SIZE = 0.30
RANDOM_STATE = 42

# avoid retrieval leakage
retrieval_cases_df, in_scope_test_cases_df = train_test_split(
    in_scope_cases_df,
    test_size=RETRIEVAL_TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
)

retrieval_cases_df = retrieval_cases_df.reset_index(drop=True)
in_scope_test_cases_df = in_scope_test_cases_df.reset_index(drop=True)

print("Original in-scope QA pairs:", in_scope_cases_df.shape)
print("Retrieval-library QA pairs:", retrieval_cases_df.shape)
print("Held-out in-scope test QA pairs:", in_scope_test_cases_df.shape)
print("Out-of-scope test QA pairs:", out_of_scope_cases_df.shape)

Original in-scope QA pairs: (50, 9)
Retrieval-library QA pairs: (35, 9)
Held-out in-scope test QA pairs: (15, 9)
Out-of-scope test QA pairs: (30, 9)


# 2. Benchmark Item Construction
In this section, each bilingual case is expanded into separate English and Dutch benchmark items that can be used directly in the chatbot pipeline.

In [ ]:
def expand_cases_to_items(
    cases_df: pd.DataFrame,
    dataset_name: str,
) -> pd.DataFrame:
    """
    Expand each bilingual case into separate English and Dutch items.
    """

    benchmark_items = []
    case_records = cases_df.to_dict(orient="records")

    # english/dutch items
    for case in case_records:
        # shared fields
        base_case_id = case["case_id"]
        context = case["context"]
        expected_answer_points = case["expected_answer_points"]
        should_not_include = case["should_not_include"]
        source_basis = case.get("source_basis", "")

        # english item
        if case["question_en"]:
            benchmark_items.append({
                "item_id": f"{base_case_id}_EN",
                "base_case_id": base_case_id,
                "dataset": dataset_name,
                "language": "en",
                "question": case["question_en"],
                "context": context,
                "reference_answer": case["reference_answer_en"],
                "expected_answer_points": expected_answer_points,
                "should_not_include": should_not_include,
                "source_basis": source_basis,
            })

        # dutch item
        if case["question_nl"]:
            benchmark_items.append({
                "item_id": f"{base_case_id}_NL",
                "base_case_id": base_case_id,
                "dataset": dataset_name,
                "language": "nl",
                "question": case["question_nl"],
                "context": context,
                "reference_answer": case["reference_answer_nl"],
                "expected_answer_points": expected_answer_points,
                "should_not_include": should_not_include,
                "source_basis": source_basis,
            })

    return pd.DataFrame(benchmark_items)

In [ ]:
case_library_df = expand_cases_to_items(
    cases_df=retrieval_cases_df,
    dataset_name="retrieval_library",
)

in_scope_test_df = expand_cases_to_items(
    cases_df=in_scope_test_cases_df,
    dataset_name="in_scope",
)

out_of_scope_test_df = expand_cases_to_items(
    cases_df=out_of_scope_cases_df,
    dataset_name="out_of_scope",
)

test_df = pd.concat(
    [in_scope_test_df, out_of_scope_test_df],
    ignore_index=True,
)

In [ ]:
print("Retrieval library shape:", case_library_df.shape)
print("Held-out in-scope test shape:", in_scope_test_df.shape)
print("Out-of-scope test shape:", out_of_scope_test_df.shape)
print("Final test set shape:", test_df.shape)

display(test_df[["item_id", "dataset", "language", "question"]].head(10))

Retrieval library shape: (70, 10)
Held-out in-scope test shape: (30, 10)
Out-of-scope test shape: (60, 10)
Final test set shape: (90, 10)


,item_id,dataset,language,question
0,EXISTING_014_EN,in_scope,en,The net-metering scheme is being phased out. H...
1,EXISTING_014_NL,in_scope,nl,De salderingsregeling wordt afgebouwd. Hoe beï...
2,NEW_024_EN,in_scope,en,What does 'building-related energy use' mean i...
3,NEW_024_NL,in_scope,nl,Wat betekent 'gebouwgebonden energieverbruik' ...
4,NEW_015_EN,in_scope,en,Should I open the windows a lot if I already h...
5,NEW_015_NL,in_scope,nl,Moet ik de ramen veel openzetten als ik al bal...
6,NEW_030_EN,in_scope,en,Should I turn off my WTW or balanced ventilati...
7,NEW_030_NL,in_scope,nl,Moet ik mijn WTW of balansventilatie uitzetten...
8,NEW_002_EN,in_scope,en,"My electricity use is high, but I don't see an..."
9,NEW_002_NL,in_scope,nl,"Mijn stroomverbruik is hoog, maar ik zie geen ..."


# 3. Case Retrieval Preparation
In this section, the case library is prepared for similarity-based retrieval. The retrieved cases will later be used as few-shot examples in the chatbot prompt.

In [ ]:
def build_retrieval_text(row: pd.Series) -> str:
    """
    Build a text field used for similarity matching.

    The retrieval text combines the user question and system-side context.
    """

    question = row["question"]
    context = row["context"]

    return f"Question: {question}. Context: {context}".strip()


retrieval_df = case_library_df.copy()
# similarity text
retrieval_df["retrieval_text"] = retrieval_df.apply(build_retrieval_text, axis=1)

# language indexes
retrieval_df_en = retrieval_df[retrieval_df["language"] == "en"].reset_index(drop=True)
retrieval_df_nl = retrieval_df[retrieval_df["language"] == "nl"].reset_index(drop=True)

print("Retrieval-ready library shape:", retrieval_df.shape)
print("English retrieval items:", retrieval_df_en.shape)
print("Dutch retrieval items:", retrieval_df_nl.shape)

display(retrieval_df[["item_id", "dataset", "language", "question", "retrieval_text"]].head(6))

Retrieval-ready library shape: (70, 11)
English retrieval items: (35, 11)
Dutch retrieval items: (35, 11)


,item_id,dataset,language,question,retrieval_text
0,EXISTING_007_EN,retrieval_library,en,Why is my energy bill higher than I expected g...,Question: Why is my energy bill higher than I ...
1,EXISTING_007_NL,retrieval_library,nl,Waarom is mijn energierekening hoger dan ik ve...,Question: Waarom is mijn energierekening hoger...
2,NEW_026_EN,retrieval_library,en,"My solar production is above expected, so why ...",Question: My solar production is above expecte...
3,NEW_026_NL,retrieval_library,nl,"Mijn zonne-opwek is hoger dan verwacht, dus wa...",Question: Mijn zonne-opwek is hoger dan verwac...
4,NEW_031_EN,retrieval_library,en,How can I keep my home less damp without using...,Question: How can I keep my home less damp wit...
5,NEW_031_NL,retrieval_library,nl,Hoe houd ik mijn woning minder vochtig zonder ...,Question: Hoe houd ik mijn woning minder vocht...


In [ ]:
# vectorizers
vectorizer_en = TfidfVectorizer()
vectorizer_nl = TfidfVectorizer()

# build tf-idf matrices
tfidf_matrix_en = vectorizer_en.fit_transform(retrieval_df_en["retrieval_text"])
tfidf_matrix_nl = vectorizer_nl.fit_transform(retrieval_df_nl["retrieval_text"])

print("English TF-IDF matrix shape:", tfidf_matrix_en.shape)
print("Dutch TF-IDF matrix shape:", tfidf_matrix_nl.shape)


def retrieve_similar_cases(
    user_question: str,
    language: str,
    top_k: int = 2,
) -> pd.DataFrame:
    """
    Retrieve the top-k most similar cases in the same language.
    """

    # choose language index
    if language == "en":
        vectorizer = vectorizer_en
        tfidf_matrix = tfidf_matrix_en
        retrieval_table = retrieval_df_en
    elif language == "nl":
        vectorizer = vectorizer_nl
        tfidf_matrix = tfidf_matrix_nl
        retrieval_table = retrieval_df_nl
    else:
        raise ValueError("Supported languages are only 'en' and 'nl'.")

    retrieval_text = f"Question: {user_question}"

    query_vector = vectorizer.transform([retrieval_text])
    # similarity scores
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()

    # top matches
    top_indices = similarities.argsort()[::-1][:top_k]

    # attach scores
    results = retrieval_table.iloc[top_indices].copy()
    results["similarity_score"] = similarities[top_indices]

    return results[
        [
            "item_id",
            "base_case_id",
            "language",
            "question",
            "context",
            "reference_answer",
            "similarity_score",
        ]
    ]

English TF-IDF matrix shape: (35, 330)
Dutch TF-IDF matrix shape: (35, 410)


In [ ]:

test_question_en = "Why is my electricity use higher at night than before?"
results_en = retrieve_similar_cases(test_question_en, language="en", top_k=2)

display(results_en)

,item_id,base_case_id,language,question,context,reference_answer,similarity_score
10,EXISTING_001_EN,EXISTING_001,en,Why is my electricity use suddenly higher at n...,Electricity use between 00:00–05:00 has been a...,Your night-time use is higher than your own ea...,0.516082
33,NEW_013_EN,NEW_013,en,Why is my hot water use much higher on weekends?,Hot-water use is about [X]% higher on Saturday...,I see that hot-water use is higher on Saturday...,0.250821


In [ ]:

test_question_nl = "Waarom is mijn elektriciteitsverbruik 's nachts hoger dan eerst?"
results_nl = retrieve_similar_cases(test_question_nl, language="nl", top_k=2)

display(results_nl)

,item_id,base_case_id,language,question,context,reference_answer,similarity_score
10,EXISTING_001_NL,EXISTING_001,nl,Waarom is mijn elektriciteitsverbruik 's nacht...,Electricity use between 00:00–05:00 has been a...,Uw nachtelijk verbruik is hoger dan uw eigen e...,0.408705
18,EXISTING_002_NL,EXISTING_002,nl,Waarom is mijn elektriciteitsverbruik deze maa...,Monthly total has increased compared to the pr...,"Deze maand heeft u [X] kWh verbruikt, vergelek...",0.346008


In [ ]:
DATA_DIR = pathlib.Path("/content")

# save retrieval tables
retrieval_df.to_csv(DATA_DIR / "retrieval_case_library.csv", index=False)
retrieval_df_en.to_csv(DATA_DIR / "retrieval_case_library_en.csv", index=False)
retrieval_df_nl.to_csv(DATA_DIR / "retrieval_case_library_nl.csv", index=False)

print("Retrieval-ready case library saved.")

Retrieval-ready case library saved.


## 4. Prompt Construction

In this section, the chatbot prompt is constructed from the available pipeline inputs.

The prompt combines:
- the current user question,
- the selected answer language,
- optional system-side or anomaly context,
- general answer guidelines,
- similar retrieved cases as few-shot examples.

This prompt will later be passed to a language model in the single-turn answer generation prototype.

In [ ]:
def format_guidelines(guideline_dict: dict) -> str:
    """
    Convert the guideline dictionary into a readable prompt section.
    """

    guideline_sections = []

    for guideline_type, guideline_texts in guideline_dict.items():
        guideline_sections.append(str(guideline_type))

        for guideline_text in guideline_texts:
            guideline_sections.append(f"- {guideline_text}")

        guideline_sections.append("")

    return "\n".join(guideline_sections).strip()

In [ ]:
import pandas as pd

def format_retrieved_cases(
    retrieved_cases: pd.DataFrame,
    similarity_threshold: float = 0.45,
) -> str:
    """
    Convert retrieved similar cases into prompt examples.

    Reference answers are only shown when the retrieved case is sufficiently similar.
    """

    examples = []

    for example_number, (_, row) in enumerate(retrieved_cases.iterrows(), start=1):
        similarity_score = float(row["similarity_score"])

        # threshold check
        include_reference_answer = similarity_score >= similarity_threshold

        # strong match
        if include_reference_answer:
            example_text = f"""
Example {example_number}
User question: {row["question"]}
System context: {row["context"]}
Answer style example: {row["reference_answer"]}
Similarity score: {similarity_score:.3f}
Important: Use this only as a style example. Do not copy facts, numbers, causes, or device-specific explanations unless the current context supports them.
""".strip()
        # weak match
        else:
            example_text = f"""
Example {example_number}
User question: {row["question"]}
System context: {row["context"]}
Similarity score: {similarity_score:.3f}
Reference answer not shown because this example is not similar enough.
""".strip()

        examples.append(example_text)

    return "\n\n".join(examples)

In [ ]:
def build_chatbot_prompt(
    user_question: str,
    language: str,
    system_context: str = "",
    top_k: int = 2,
) -> str:
    """
    Build a strict single-turn RAG chatbot prompt.
    """

    if language not in ["en", "nl"]:
        raise ValueError("Supported languages are only 'en' and 'nl'.")

    language_name = {
        "en": "English",
        "nl": "Dutch",
    }[language]

    # retrieve examples
    retrieved_cases = retrieve_similar_cases(
        user_question=user_question,
        language=language,
        top_k=top_k,
    )

    guidelines_text = format_guidelines(guideline_dict)

    examples_text = format_retrieved_cases(
        retrieved_cases=retrieved_cases,
        similarity_threshold=0.45,
    )

    # context fallback
    if not system_context:
        system_context = "No additional system-side context is available for this question."

    prompt = f"""
You are a BeNext energy support assistant for household energy monitoring.

Your task is to answer only questions about household energy use, energy monitoring, energy devices, and anomaly follow-up.
The answer must be written for non-technical household users.

Answer language:
{language_name}

General answer guidelines:
{guidelines_text}

Similar example cases:
{examples_text}

Important rule about examples:
The similar example cases are only examples of tone and structure. Do not copy their facts, numbers, device names, causes, or conclusions unless the current system-side context supports them.

Current system-side context:
{system_context}

Current user question:
{user_question}

Answer requirements:
- Output only the final user-facing chatbot answer.
- Do not include headings, bullet points, markdown, labels, or step-by-step reasoning.
- Do not start with phrases such as "Sure", "Let's break down", "Here is the answer", or similar.
- Answer in {language_name}.
- Write 2 to 4 short sentences.
- Use simple, non-technical language.
- Use only the current system-side context as evidence.
- Do not invent causes.
- If the exact cause is not confirmed, say that clearly.
- Give at most one practical next step.
- Avoid alarmist wording.
- Never mention Qwen, Hugging Face, OpenAI, LLM, AI model, prompt, retrieved examples, benchmark data, or internal pipeline.
- If asked what you are, refer to yourself only as a BeNext energy support assistant.
- Do not reuse a retrieved example answer if it conflicts with the current question or context.
- First identify the observed pattern from the current context. Then explain only that pattern.

Final chatbot answer only:
""".strip()

    return prompt

## 5. Single-Turn Answer Generation Prototype

In this section, the chatbot pipeline is tested for one question at a time.

The prototype takes a user question, a language, and optional system-side context. It then builds a prompt using the retrieved few-shot examples and guidelines from the previous sections. The prompt is passed to a language model, which generates a single chatbot answer.

This version is single-turn only. It does not yet handle clarification questions or conversation history.

### 5.1 Automatic Language Detection

This section adds automatic language detection for live-style chatbot use.

Two language detection methods are implemented:

1. Lingua-based detection:
   - uses the `lingua-language-detector` Python package,
   - restricted to English and Dutch,
   - fast and suitable for short user questions.

2. Prompt-based detection:
   - uses the loaded language model as a routing classifier,
   - returns a structured JSON label,
   - useful as an experimental comparison.

In [ ]:
# lingua setup
lingua_detector = (
    LanguageDetectorBuilder
    .from_languages(Language.ENGLISH, Language.DUTCH)
    .build()
)

In [ ]:
def detect_language(user_question: str) -> str:
    """
    Detect whether the user question is English or Dutch using Lingua.

    Returns:
    - 'en'
    - 'nl'
    """

    detected_language = lingua_detector.detect_language_of(user_question)

    if detected_language == Language.DUTCH:
        return "nl"

    if detected_language == Language.ENGLISH:
        return "en"

    # default language
    return "en"

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bnb.__version__)

CUDA available: True
GPU: Tesla T4
transformers: 4.45.0
accelerate: 1.13.0
bitsandbytes: 0.49.2


In [ ]:
# hf login

from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Successfully logged in to Hugging Face Hub.")
else:
    print("HF_TOKEN not found in Colab secrets. Please add it.")

Successfully logged in to Hugging Face Hub.


In [ ]:
# tested models
MODEL_IDS_TO_TEST = [
    "Qwen/Qwen2.5-7B-Instruct",
    "Qwen/Qwen2.5-14B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B",
]

# fixed router
ROUTER_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

In [ ]:
def generate_scope_labels_for_dataframe(
    input_df: pd.DataFrame,
    router_model_id: str,
) -> pd.DataFrame:
    """
    Precompute scope labels for all rows using one fixed router model.

    This allows all later answer-generation models to use the same
    scope decisions.
    """

    global tokenizer
    global model
    global device
    global CURRENT_MODEL_ID

    # input copy
    routed_df = input_df.copy()

    print("=" * 80)
    print(f"Loading fixed router model: {router_model_id}")
    print("=" * 80)

    CURRENT_MODEL_ID = router_model_id
    # load router
    tokenizer, model, device = load_generation_model(router_model_id)

    # routing buffers
    detected_scopes = []
    detected_languages = []
    routing_latencies = []

    total_items = len(routed_df)

    for row_number, (_, row) in enumerate(routed_df.iterrows(), start=1):
        question = row.get("question", "")

        print(f"Routing {row_number}/{total_items}: {row.get('item_id', f'row_{row_number}')}")

        start_time = time.perf_counter()

        # route labels
        detected_language = detect_language(question)
        detected_scope = detect_scope(question)

        end_time = time.perf_counter()

        detected_languages.append(detected_language)
        detected_scopes.append(detected_scope)
        routing_latencies.append(end_time - start_time)

    # cache routing
    routed_df["cached_detected_language"] = detected_languages
    routed_df["cached_detected_scope"] = detected_scopes
    routed_df["routing_model_id"] = router_model_id
    routed_df["routing_latency_seconds"] = routing_latencies

    # free gpu
    unload_generation_model()

    print("Routing finished.")
    # scope summary
    print("Cached scope distribution:")
    print(routed_df["cached_detected_scope"].value_counts(dropna=False))

    return routed_df

In [ ]:
def load_generation_model(model_id: str):
    """
    Load one instruction-tuned model and tokenizer in 4-bit quantization.
    """

    print("=" * 80)
    print(f"Loading model: {model_id}")
    print("=" * 80)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # quantized config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # inference mode
    model.eval()

    print("Model loaded:", model_id)

    return tokenizer, model, device

In [ ]:
import gc

def unload_generation_model():
    """
    Clear the currently loaded model from GPU memory.
    """

    global model
    global tokenizer

    try:
        del model
    except NameError:
        pass

    try:
        del tokenizer
    except NameError:
        pass

    gc.collect()

    if torch.cuda.is_available():
        # cuda cleanup
        torch.cuda.empty_cache()

    print("Model unloaded and GPU cache cleared.")

In [ ]:
def generate_text_with_model(
    prompt: str,
    max_new_tokens: int = 120,
) -> str:
    """
    Generate an answer from a completed chatbot prompt using deterministic decoding.
    """

    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    try:
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        formatted_prompt = prompt

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    ).to(device)

    input_length = inputs["input_ids"].shape[-1]

    generation_config = GenerationConfig.from_model_config(model.config)
    generation_config.do_sample = False
    generation_config.pad_token_id = tokenizer.eos_token_id
    generation_config.eos_token_id = tokenizer.eos_token_id

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            generation_config=generation_config,
            max_new_tokens=max_new_tokens,
        )

    # new tokens only
    generated_ids = output_ids[0][input_length:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    return answer

In [ ]:
# answer cleanup

def clean_generated_answer(answer: str) -> str:
    """
    Remove common unwanted openings, headings, markdown, and model-style wording.
    This is a light post-processing step. It improves formatting, but it does not
    replace better prompting or scope detection.
    """

    if not isinstance(answer, str):
        return ""

    cleaned_answer = answer.strip()

    # opening phrases
    unwanted_prefixes = [
        "Sure!",
        "Sure,",
        "Of course!",
        "Of course,",
        "Let's break down",
        "Let’s break down",
        "Here is the answer:",
        "Here is the response:",
        "Final answer:",
        "Final chatbot answer:",
        "Answer:",
        "Hier is de juiste antwoord:",
        "Hier is het juiste antwoord:",
        "Hier is het antwoord:",
        "Natuurlijk!",
        "Natuurlijk,",
        "Zeker!",
        "Zeker,",
    ]

    for prefix in unwanted_prefixes:
        if cleaned_answer.lower().startswith(prefix.lower()):
            cleaned_answer = cleaned_answer[len(prefix):].strip()

    # unwanted headings
    unwanted_headings = [
        "### System Context:",
        "### Reasoning:",
        "### Explanation:",
        "### Answer:",
        "## System Context:",
        "## Reasoning:",
        "## Explanation:",
        "## Answer:",
        "System Context:",
        "Reasoning:",
        "Explanation:",
    ]

    for heading in unwanted_headings:
        cleaned_answer = cleaned_answer.replace(heading, "").strip()

    # internal terms
    forbidden_identity_terms = [
        "Qwen",
        "Hugging Face",
        "OpenAI",
        "LLM",
        "large language model",
        "AI model",
        "language model",
        "prompt",
        "retrieved examples",
        "benchmark",
        "internal pipeline",
    ]

    for term in forbidden_identity_terms:
        cleaned_answer = cleaned_answer.replace(term, "BeNext energy support assistant")

    # remove code fences
    cleaned_answer = cleaned_answer.replace("```", "").strip()

    # compact spaces
    cleaned_answer = " ".join(cleaned_answer.split())

    return cleaned_answer

In [ ]:
def parse_json_from_text(raw_output: str) -> dict:
    """
    Try to parse a JSON object from model output.
    If the model returns extra text, extract the first JSON-like block.
    """

    try:
        return json.loads(raw_output)
    except Exception:
        pass

    try:
        start_index = raw_output.find("{")
        end_index = raw_output.rfind("}") + 1

        if start_index != -1 and end_index != -1:
            json_text = raw_output[start_index:end_index]
            return json.loads(json_text)
    except Exception:
        pass

    # empty fallback
    return {}

In [ ]:
def detect_scope(user_question: str) -> str:
    """
    Detect whether a user question is in scope using the language model.

    Returns:
    - 'in_scope'
    - 'out_of_scope'
    """

    # classifier prompt
    scope_prompt = f"""
You are a routing classifier for a BeNext household energy monitoring chatbot.

Classify the user question as exactly one of these labels:

1. in_scope
The question is about household energy monitoring, electricity use, energy consumption, solar production, grid export, heat pump behaviour, ventilation, water use, device-related energy behaviour, energy graphs, peaks, anomalies, or follow-up recommendations based on system readings.

2. out_of_scope
The question is about anything else. This includes recipes, stocks, medical advice, politics, travel, coding, movies, translation, jokes, Wi-Fi/router support, account/login support, legal advice, definitive billing advice, tariff advice, compensation advice, device repair instructions, installer-level troubleshooting, or housing-provider disputes.

User question:
{user_question}

Return only valid JSON with this exact structure:
{{
  "scope": "in_scope",
  "reason": "Brief reason."
}}
""".strip()

    raw_output = generate_text_with_model(
        prompt=scope_prompt,
        max_new_tokens=80,
    )

    parsed_output = parse_json_from_text(raw_output)

    scope = parsed_output.get("scope", "").strip().lower()

    if scope in ["in_scope", "out_of_scope"]:
        return scope

    # safe refusal
    return "out_of_scope"

In [ ]:
def generate_single_turn_answer(
    user_question: str,
    language: str = "",
    system_context: str = "",
    top_k: int = 2,
    max_new_tokens: int = 220,
    forced_scope: str = "",
) -> dict:
    """
    Generate one chatbot answer for one user question.

    Language is detected with Lingua when not provided.
    Scope is detected with the prompt-based scope classifier unless
    a forced scope label is provided.
    """

    start_time = time.perf_counter()

    if not language:
        language = detect_language(user_question)

    detected_language = language

    # cached routing
    if forced_scope:
        detected_scope = forced_scope
        scope_detection_method = "cached_qwen7b_router"
    else:
        detected_scope = detect_scope(user_question)
        scope_detection_method = "prompt_based"

    # refusal path
    if detected_scope == "out_of_scope":
        answer = get_out_of_scope_response(language)
        end_time = time.perf_counter()

        return {
            "user_question": user_question,
            "language": language,
            "system_context": system_context,
            "top_k": top_k,
            "prompt": "",
            "generated_answer": answer,
            "latency_seconds": end_time - start_time,
            "detected_scope": detected_scope,
            "scope_detection_method": scope_detection_method,
            "detected_language": detected_language,
            "language_detection_method": "lingua",
        }

    # rag prompt
    prompt = build_chatbot_prompt(
        user_question=user_question,
        language=language,
        system_context=system_context,
        top_k=top_k,
    )

    answer = generate_text_with_model(
        prompt=prompt,
        max_new_tokens=max_new_tokens,
    )

    answer = clean_generated_answer(answer)

    end_time = time.perf_counter()

    return {
        "user_question": user_question,
        "language": language,
        "system_context": system_context,
        "top_k": top_k,
        "prompt": prompt,
        "generated_answer": answer,
        "latency_seconds": end_time - start_time,
        "detected_scope": detected_scope,
        "scope_detection_method": scope_detection_method,
        "detected_language": detected_language,
        "language_detection_method": "lingua",
    }

In [ ]:

def get_out_of_scope_response(language: str) -> str:
    """
    Return a fixed refusal response for out-of-scope questions.
    This avoids sending unrelated questions to the LLM.
    """

    if language == "nl":
        return (
            "Ik kan alleen helpen met vragen over energiemonitoring in huis, "
            "energieverbruik, zonne-energie, warmtepompen, ventilatie en afwijkende energiepatronen. "
            "Stel gerust een vraag over uw energiegegevens of systeemmetingen."
        )

    return (
        "I can only help with questions about household energy monitoring, "
        "energy use, solar production, heat pumps, ventilation, and related anomalies. "
        "Please ask a question about your energy data or system readings."
    )

## 5.2 QA Approach Comparison Setup

This section defines three QA approaches for the thesis comparison:

1. Closed-book QA: the model answers using only the user question, system-side context, and guidelines.
2. Retrieval-based QA: the system retrieves the most similar case and returns its reference answer as a nearest-neighbour baseline.
3. Retrieval-augmented generation (RAG): the system retrieves similar cases and passes them into the prompt before generation.

Out-of-scope questions are handled by the same deterministic refusal layer for all approaches.

In [ ]:
def build_closed_book_prompt(
    user_question: str,
    language: str,
    system_context: str = "",
) -> str:
    """
    Build a closed-book QA prompt.
    """

    if language not in ["en", "nl"]:
        raise ValueError("Supported languages are only 'en' and 'nl'.")

    language_name = {
        "en": "English",
        "nl": "Dutch",
    }[language]

    guidelines_text = format_guidelines(guideline_dict)

    # context fallback
    if not system_context:
        system_context = "No additional system-side context is available for this question."

    prompt = f"""
You are a BeNext energy support assistant for household energy monitoring.

This is a closed-book QA setting. Do not use retrieved examples or external documents.
Answer only the current user question using the current system-side context and the answer guidelines.

Answer language:
{language_name}

General answer guidelines:
{guidelines_text}

Current system-side context:
{system_context}

Current user question:
{user_question}

Answer requirements:
- Output only the final user-facing chatbot answer.
- Do not include headings, bullet points, markdown, labels, or step-by-step reasoning.
- Do not start with phrases such as "Sure", "Let's break down", "Here is the answer", or similar.
- Answer in {language_name}.
- Write 2 to 4 short sentences.
- Use simple, non-technical language.
- Use only the current system-side context as evidence.
- Do not invent causes.
- If the exact cause is not confirmed, say that clearly.
- Give at most one practical next step.
- Avoid alarmist wording.
- Never mention Qwen, Hugging Face, OpenAI, LLM, AI model, prompt, benchmark data, or internal pipeline.
- If asked what you are, refer to yourself only as a BeNext energy support assistant.

Final chatbot answer only:
""".strip()

    return prompt

In [ ]:
def generate_closed_book_answer(
    user_question: str,
    language: str = "",
    system_context: str = "",
    max_new_tokens: int = 90,
    forced_scope: str = "",
) -> dict:
    """
    Generate an answer using the closed-book QA approach.
    """

    start_time = time.perf_counter()

    if not language:
        language = detect_language(user_question)

    detected_language = language
    # cached routing
    if forced_scope:
        detected_scope = forced_scope
        scope_detection_method = "cached_qwen7b_router"
    else:
        detected_scope = detect_scope(user_question)
        scope_detection_method = "prompt_based"

    # refusal path
    if detected_scope == "out_of_scope":
        answer = get_out_of_scope_response(language)
        end_time = time.perf_counter()

        return {
            "approach": "closed_book",
            "user_question": user_question,
            "language": language,
            "system_context": system_context,
            "top_k": 0,
            "prompt": "",
            "generated_answer": answer,
            "latency_seconds": end_time - start_time,
            "detected_scope": detected_scope,
            "retrieved_item_ids": "",
            "retrieval_similarity_scores": "",
            "scope_detection_method": scope_detection_method,
            "detected_language": detected_language,
            "language_detection_method": "lingua",
        }

    prompt = build_closed_book_prompt(
        user_question=user_question,
        language=language,
        system_context=system_context,
    )

    answer = generate_text_with_model(
        prompt=prompt,
        max_new_tokens=max_new_tokens,
    )

    answer = clean_generated_answer(answer)

    end_time = time.perf_counter()

    return {
        "approach": "closed_book",
        "user_question": user_question,
        "language": language,
        "system_context": system_context,
        "top_k": 0,
        "prompt": prompt,
        "generated_answer": answer,
        "latency_seconds": end_time - start_time,
        "detected_scope": detected_scope,
        "retrieved_item_ids": "",
        "retrieval_similarity_scores": "",
        "scope_detection_method": scope_detection_method,
        "detected_language": detected_language,
        "language_detection_method": "lingua",
    }

In [ ]:
def generate_retrieval_based_answer(
    user_question: str,
    language: str = "",
    system_context: str = "",
    top_k: int = 1,
    similarity_threshold: float = 0.20,
    forced_scope: str = "",
) -> dict:
    """
    Generate an answer using a retrieval-based QA baseline.

    This approach does not generate a new answer with the LLM.
    It retrieves the most similar case and returns its reference answer.
    """

    start_time = time.perf_counter()

    if not language:
        language = detect_language(user_question)

    detected_language = language
    # cached routing
    if forced_scope:
        detected_scope = forced_scope
        scope_detection_method = "cached_qwen7b_router"
    else:
        detected_scope = detect_scope(user_question)
        scope_detection_method = "prompt_based"

    # refusal path
    if detected_scope == "out_of_scope":
        answer = get_out_of_scope_response(language)
        end_time = time.perf_counter()

        return {
            "approach": "retrieval_based",
            "user_question": user_question,
            "language": language,
            "system_context": system_context,
            "top_k": top_k,
            "prompt": "",
            "generated_answer": answer,
            "latency_seconds": end_time - start_time,
            "detected_scope": detected_scope,
            "retrieved_item_ids": "",
            "retrieval_similarity_scores": "",
            "scope_detection_method": scope_detection_method,
            "detected_language": detected_language,
            "language_detection_method": "lingua",
        }

    # nearest cases
    retrieved_cases = retrieve_similar_cases(
        user_question=user_question,
        language=language,
        top_k=top_k,
    )

    best_case = retrieved_cases.iloc[0]
    best_similarity = float(best_case["similarity_score"])

    # fallback threshold
    if best_similarity < similarity_threshold:
        if language == "nl":
            answer = (
                "Ik kan op basis van de beschikbare gegevens geen voldoende vergelijkbaar voorbeeld vinden. "
                "De situatie kan wel wijzen op een afwijkend energiepatroon, maar de exacte oorzaak is niet bevestigd. "
                "Controleer de grafiek of systeemmetingen opnieuw als dit patroon aanhoudt."
            )
        else:
            answer = (
                "I cannot find a sufficiently similar example based on the available information. "
                "The situation may indicate an unusual energy pattern, but the exact cause is not confirmed. "
                "Check the graph or system readings again if the pattern continues."
            )
    else:
        # reference answer
        answer = best_case["reference_answer"]

    answer = clean_generated_answer(answer)

    end_time = time.perf_counter()

    return {
        "approach": "retrieval_based",
        "user_question": user_question,
        "language": language,
        "system_context": system_context,
        "top_k": top_k,
        "prompt": "",
        "generated_answer": answer,
        "latency_seconds": end_time - start_time,
        "detected_scope": detected_scope,
        "retrieved_item_ids": "; ".join(retrieved_cases["item_id"].astype(str).tolist()),
        "retrieval_similarity_scores": "; ".join(
            retrieved_cases["similarity_score"].round(3).astype(str).tolist()
        ),
        "scope_detection_method": scope_detection_method,
        "detected_language": detected_language,
        "language_detection_method": "lingua",
    }

In [ ]:
def generate_rag_answer(
    user_question: str,
    language: str = "",
    system_context: str = "",
    top_k: int = 2,
    max_new_tokens: int = 90,
    forced_scope: str = "",
) -> dict:
    """
    Generate an answer using the RAG approach.
    """

    result = generate_single_turn_answer(
        user_question=user_question,
        language=language,
        system_context=system_context,
        top_k=top_k,
        max_new_tokens=max_new_tokens,
        forced_scope=forced_scope,
    )

    result["approach"] = "rag"

    resolved_language = (
        result.get("detected_language", "")
        or result.get("language", "")
        or language
    )

    # retrieval metadata
    if result.get("detected_scope") == "in_scope" and resolved_language in ["en", "nl"]:
        retrieved_cases = retrieve_similar_cases(
            user_question=user_question,
            language=resolved_language,
            top_k=top_k,
        )
    else:
        retrieved_cases = pd.DataFrame()

    if len(retrieved_cases) > 0:
        result["retrieved_item_ids"] = "; ".join(
            retrieved_cases["item_id"].astype(str).tolist()
        )
        result["retrieval_similarity_scores"] = "; ".join(
            retrieved_cases["similarity_score"].round(3).astype(str).tolist()
        )
    else:
        result["retrieved_item_ids"] = ""
        result["retrieval_similarity_scores"] = ""

    return result

In [ ]:
def generate_answer_by_approach(
    user_question: str,
    language: str = "",
    system_context: str = "",
    approach: str = "rag",
    top_k: int = 2,
    max_new_tokens: int = 90,
    forced_scope: str = "",
) -> dict:
    """
    Generate an answer using one of the thesis QA approaches.
    """

    if approach == "closed_book":
        return generate_closed_book_answer(
            user_question=user_question,
            language=language,
            system_context=system_context,
            max_new_tokens=max_new_tokens,
            forced_scope=forced_scope,
        )

    if approach == "retrieval_based":
        return generate_retrieval_based_answer(
            user_question=user_question,
            language=language,
            system_context=system_context,
            top_k=1,
            forced_scope=forced_scope,
        )

    if approach == "rag":
        return generate_rag_answer(
            user_question=user_question,
            language=language,
            system_context=system_context,
            top_k=top_k,
            max_new_tokens=max_new_tokens,
            forced_scope=forced_scope,
        )

    raise ValueError("Supported approaches are: 'closed_book', 'retrieval_based', and 'rag'.")

## 6. Batch Answer Generation

In this section, the single-turn chatbot pipeline is applied to multiple benchmark items.

The goal is to generate chatbot answers for the benchmark questions and store the outputs in a structured results table. This prepares the notebook for later evaluation, where generated answers can be compared across languages, question types, and evaluation metrics.


In [ ]:
# batch helpers

def safe_serialize(value):
    """
    Convert lists or dictionaries into JSON strings before saving to CSV.
    Other values are returned unchanged.
    """
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)
    return value

In [ ]:
def generate_answers_for_dataframe(
    input_df: pd.DataFrame,
    approach: str = "rag",
    top_k: int = 2,
    max_new_tokens: int = 90,
    store_prompt: bool = False,
) -> pd.DataFrame:
    """
    Generate chatbot answers for all rows in a dataframe using one QA approach.

    Language is always detected with Lingua.
    Scope is always detected with the prompt-based classifier.
    """

    generated_rows = []

    total_items = len(input_df)

    print(
        f"Starting batch generation for {total_items} items | "
        f"approach={approach}, "
        f"language_detection=lingua, "
        f"scope_detection=prompt_based"
    )

    for row_number, (_, row) in enumerate(input_df.iterrows(), start=1):
        item_id = row.get("item_id", f"row_{row_number}")

        true_language = row.get("language", "")
        question = row.get("question", "")
        context = row.get("context", "")
        forced_scope = row.get("cached_detected_scope", "")

        print(
            f"\nGenerating {row_number}/{total_items}: {item_id} "
            f"(true_language={true_language}, approach={approach})"
        )

        try:
            result = generate_answer_by_approach(
                user_question=question,
                language="",  # force lingua
                system_context=context,
                approach=approach,
                top_k=top_k,
                max_new_tokens=max_new_tokens,
                forced_scope=forced_scope,
            )

            generated_answer = result["generated_answer"]
            latency_seconds = result["latency_seconds"]
            prompt = result.get("prompt", "")
            error_message = ""

        except Exception as error:
            result = {}
            generated_answer = ""
            latency_seconds = None
            prompt = ""
            error_message = str(error)

            print(f"Error for {item_id}: {error_message}")

        detected_language = result.get("detected_language", "")

        output_row = {
            "item_id": item_id,
            "base_case_id": row.get("base_case_id", ""),
            "dataset": row.get("dataset", ""),
            "source_basis": row.get("source_basis", ""),
            "approach": approach,

            "true_language": true_language,
            "language": result.get("language", detected_language),
            "detected_language": detected_language,
            "language_detection_method": "lingua",
            "language_detection_correct": detected_language == true_language,

            "question": question,
            "context": context,
            "reference_answer": row.get("reference_answer", ""),
            "expected_answer_points": safe_serialize(row.get("expected_answer_points", "")),
            "should_not_include": safe_serialize(row.get("should_not_include", "")),

            "generated_answer": generated_answer,
            "latency_seconds": latency_seconds,
            "pipeline_model_id": CURRENT_MODEL_ID,
            "answer_model_id": CURRENT_MODEL_ID if approach in ["closed_book", "rag"] else "retrieval_only",
            "routing_model_id": row.get("routing_model_id", ROUTER_MODEL_ID),
            "top_k": top_k if approach == "rag" else (1 if approach == "retrieval_based" else 0),
            "max_new_tokens": max_new_tokens if approach in ["closed_book", "rag"] else 0,
            "generation_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "error_message": error_message,

            "detected_scope": result.get("detected_scope", ""),
            "scope_detection_method": result.get("scope_detection_method", ""),
            "cached_detected_scope": row.get("cached_detected_scope", ""),
            "retrieved_item_ids": result.get("retrieved_item_ids", ""),
            "retrieval_similarity_scores": result.get("retrieval_similarity_scores", ""),
        }

        if store_prompt:
            output_row["prompt"] = prompt

        generated_rows.append(output_row)

    results_df = pd.DataFrame(generated_rows)

    print("\nBatch generation finished.")
    print("Generated results shape:", results_df.shape)

    return results_df

In [ ]:
def generate_approach_comparison_results(
    input_df: pd.DataFrame,
    approaches: list[str],
    top_k: int = 2,
    max_new_tokens: int = 90,
    store_prompt: bool = False,
) -> pd.DataFrame:
    """
    Run multiple QA approaches on the same dataframe and combine the results.

    Fixed routing setup:
    - language detection: Lingua
    - scope detection: prompt-based
    """

    all_results = []

    for approach in approaches:
        print("=" * 80)
        print(f"Running approach: {approach} | Language: lingua | Scope: prompt_based")
        print("=" * 80)

        approach_results = generate_answers_for_dataframe(
            input_df=input_df,
            approach=approach,
            top_k=top_k,
            max_new_tokens=max_new_tokens,
            store_prompt=store_prompt,
        )

        all_results.append(approach_results)

    comparison_results = pd.concat(all_results, ignore_index=True)

    print("\nAll approach runs finished.")
    print("Comparison results shape:", comparison_results.shape)

    return comparison_results

In [ ]:
def generate_multi_model_comparison_results(
    input_df: pd.DataFrame,
    model_ids: list[str],
    approaches: list[str],
    top_k: int = 2,
    max_new_tokens: int = 90,
    store_prompt: bool = False,
) -> pd.DataFrame:
    """
    Run the full QA comparison for multiple models.

    For each model:
    - load model and tokenizer,
    - run closed-book, retrieval-based, and RAG,
    - save intermediate results,
    - unload model from GPU memory.
    """

    global tokenizer
    global model
    global device
    global CURRENT_MODEL_ID

    all_model_results = []

    for model_id in model_ids:
        CURRENT_MODEL_ID = model_id

        tokenizer, model, device = load_generation_model(model_id)

        model_results = generate_approach_comparison_results(
            input_df=input_df,
            approaches=approaches,
            top_k=top_k,
            max_new_tokens=max_new_tokens,
            store_prompt=store_prompt,
        )

        model_results["pipeline_model_id"] = model_id

        output_path = DATA_DIR / f"qa_results_{model_id.replace('/', '_')}.csv"
        model_results.to_csv(output_path, index=False)

        print("Saved model results to:")
        print(output_path)

        all_model_results.append(model_results)

        # release memory
        unload_generation_model()

    combined_results = pd.concat(all_model_results, ignore_index=True)

    combined_output_path = DATA_DIR / "qa_multi_model_comparison_results.csv"
    combined_results.to_csv(combined_output_path, index=False)

    print("All model runs finished.")
    print("Combined results shape:", combined_results.shape)
    print("Combined results saved to:")
    print(combined_output_path)

    return combined_results

In [ ]:
# cached routing run
routed_test_df = generate_scope_labels_for_dataframe(
    input_df=test_df,
    router_model_id=ROUTER_MODEL_ID,
)

print("Routed test set shape:", routed_test_df.shape)

display(
    routed_test_df[
        [
            "item_id",
            "dataset",
            "language",
            "question",
            "cached_detected_language",
            "cached_detected_scope",
            "routing_model_id",
        ]
    ].head(10)
)

Loading fixed router model: Qwen/Qwen2.5-7B-Instruct
Loading model: Qwen/Qwen2.5-7B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-7B-Instruct
Routing 1/90: EXISTING_014_EN


Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Routing 2/90: EXISTING_014_NL
Routing 3/90: NEW_024_EN
Routing 4/90: NEW_024_NL
Routing 5/90: NEW_015_EN
Routing 6/90: NEW_015_NL
Routing 7/90: NEW_030_EN
Routing 8/90: NEW_030_NL
Routing 9/90: NEW_002_EN
Routing 10/90: NEW_002_NL
Routing 11/90: NEW_033_EN
Routing 12/90: NEW_033_NL
Routing 13/90: NEW_011_EN
Routing 14/90: NEW_011_NL
Routing 15/90: NEW_010_EN
Routing 16/90: NEW_010_NL
Routing 17/90: NEW_017_EN
Routing 18/90: NEW_017_NL
Routing 19/90: NEW_004_EN
Routing 20/90: NEW_004_NL
Routing 21/90: EXISTING_013_EN
Routing 22/90: EXISTING_013_NL
Routing 23/90: EXISTING_005_EN
Routing 24/90: EXISTING_005_NL
Routing 25/90: NEW_022_EN
Routing 26/90: NEW_022_NL
Routing 27/90: EXISTING_009_EN
Routing 28/90: EXISTING_009_NL
Routing 29/90: EXISTING_004_EN
Routing 30/90: EXISTING_004_NL
Routing 31/90: OOS_001_EN
Routing 32/90: OOS_001_NL
Routing 33/90: OOS_002_EN
Routing 34/90: OOS_002_NL
Routing 35/90: OOS_003_EN
Routing 36/90: OOS_003_NL
Routing 37/90: OOS_004_EN
Routing 38/90: OOS_004_NL
R

,item_id,dataset,language,question,cached_detected_language,cached_detected_scope,routing_model_id
0,EXISTING_014_EN,in_scope,en,The net-metering scheme is being phased out. H...,en,out_of_scope,Qwen/Qwen2.5-7B-Instruct
1,EXISTING_014_NL,in_scope,nl,De salderingsregeling wordt afgebouwd. Hoe beï...,nl,out_of_scope,Qwen/Qwen2.5-7B-Instruct
2,NEW_024_EN,in_scope,en,What does 'building-related energy use' mean i...,en,in_scope,Qwen/Qwen2.5-7B-Instruct
3,NEW_024_NL,in_scope,nl,Wat betekent 'gebouwgebonden energieverbruik' ...,nl,out_of_scope,Qwen/Qwen2.5-7B-Instruct
4,NEW_015_EN,in_scope,en,Should I open the windows a lot if I already h...,en,in_scope,Qwen/Qwen2.5-7B-Instruct
5,NEW_015_NL,in_scope,nl,Moet ik de ramen veel openzetten als ik al bal...,nl,in_scope,Qwen/Qwen2.5-7B-Instruct
6,NEW_030_EN,in_scope,en,Should I turn off my WTW or balanced ventilati...,en,in_scope,Qwen/Qwen2.5-7B-Instruct
7,NEW_030_NL,in_scope,nl,Moet ik mijn WTW of balansventilatie uitzetten...,nl,in_scope,Qwen/Qwen2.5-7B-Instruct
8,NEW_002_EN,in_scope,en,"My electricity use is high, but I don't see an...",en,in_scope,Qwen/Qwen2.5-7B-Instruct
9,NEW_002_NL,in_scope,nl,"Mijn stroomverbruik is hoog, maar ik zie geen ...",nl,in_scope,Qwen/Qwen2.5-7B-Instruct


In [ ]:
approaches_to_test = [
    "closed_book",
    "retrieval_based",
    "rag",
]

multi_model_results = generate_multi_model_comparison_results(
    input_df=routed_test_df,
    model_ids=MODEL_IDS_TO_TEST,
    approaches=approaches_to_test,
    top_k=2,
    max_new_tokens=90,
    store_prompt=False,
)

Loading model: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...


Generating 59/90: OOS_015_EN (true_language=en, approach=rag)

Generating 60/90: OOS_015_NL (true_language=nl, approach=rag)

Generating 61/90: OOS_016_EN (true_language=en, approach=rag)

Generating 62/90: OOS_016_NL (true_language=nl, approach=rag)

Generating 63/90: OOS_017_EN (true_language=en, approach=rag)

Generating 64/90: OOS_017_NL (true_language=nl, approach=rag)

Generating 65/90: OOS_018_EN (true_language=en, approach=rag)

Generating 66/90: OOS_018_NL (true_language=nl, approach=rag)

Generating 67/90: OOS_019_EN (true_language=en, approach=rag)

Generating 68/90: OOS_019_NL (true_language=nl, approach=rag)

Generating 69/90: OOS_020_EN (true_language=en, approach=rag)

Generating 70/90: OOS_020_NL (true_language=nl, approach=rag)

Generating 71/90: OOS_021_EN (true_language=en, approach=rag)

Generating 72/90: OOS_021_NL (true_language=nl, approach=rag)

Generating 73/90: OOS_022_EN (true_language=en, approach=rag)

Generating 74/90: OOS_022_NL (true_language=nl, approa

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Generating 80/90: OOS_025_NL (true_language=nl, approach=rag)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Generating 81/90: OOS_026_EN (true_language=en, approach=rag)

Generating 82/90: OOS_026_NL (true_language=nl, approach=rag)

Generating 83/90: OOS_027_EN (true_language=en, approach=rag)

Generating 84/90: OOS_027_NL (true_language=nl, approach=rag)

Generating 85/90: OOS_028_EN (true_language=en, approach=rag)

Generating 86/90: OOS_028_NL (true_language=nl, approach=rag)

Generating 87/90: OOS_029_EN (true_language=en, approach=rag)

Generating 88/90: OOS_029_NL (true_language=nl, approach=rag)

Generating 89/90: OOS_030_EN (true_language=en, approach=rag)

Generating 90/90: OOS_030_NL (true_language=nl, approach=rag)

Batch generation finished.
Generated results shape: (90, 29)

All approach runs finished.
Comparison results shape: (270, 29)
Saved model results to:
/content/qa_results_deepseek-ai_DeepSeek-R1-Distill-Qwen-14B.csv
Model unloaded and GPU cache cleared.
All model runs finished.
Combined results shape: (540, 29)
Combined results saved to:
/content/qa_multi_model_compar

In [ ]:
print("Expected rows:", len(test_df) * len(approaches_to_test) * len(MODEL_IDS_TO_TEST))
print("Actual rows:", len(multi_model_results))

Expected rows: 540
Actual rows: 540


In [ ]:
# save final results
comparison_output_path = DATA_DIR / "qa_approach_comparison_results.csv"

multi_model_results.to_csv(comparison_output_path, index=False)

print("QA approach comparison results saved to:")
print(comparison_output_path)

QA approach comparison results saved to:
/content/qa_approach_comparison_results.csv


In [ ]:
summary_df = (
    multi_model_results
    .groupby(["approach", "dataset", "true_language"])
    .agg(
        n_items=("item_id", "count"),
        n_errors=("error_message", lambda x: (x != "").sum()),
        avg_latency_seconds=("latency_seconds", "mean"),
        language_accuracy=("language_detection_correct", "mean"),
    )
    .reset_index()
)

display(summary_df)

,approach,dataset,true_language,n_items,n_errors,avg_latency_seconds,language_accuracy
0,closed_book,in_scope,en,30,0,11.414873,1.0
1,closed_book,in_scope,nl,30,0,10.553549,1.0
2,closed_book,out_of_scope,en,60,0,0.812264,1.0
3,closed_book,out_of_scope,nl,60,0,0.820179,1.0
4,rag,in_scope,en,30,0,12.003215,1.0
5,rag,in_scope,nl,30,0,11.188798,1.0
6,rag,out_of_scope,en,60,0,0.856893,1.0
7,rag,out_of_scope,nl,60,0,0.863697,1.0
8,retrieval_based,in_scope,en,30,0,0.002193,1.0
9,retrieval_based,in_scope,nl,30,0,0.001883,1.0


In [ ]:
model_summary = (
    multi_model_results
    .groupby(["pipeline_model_id", "approach"])
    .agg(
        n_items=("item_id", "count"),
        n_errors=("error_message", lambda x: (x != "").sum()),
        avg_latency_seconds=("latency_seconds", "mean"),
        language_accuracy=("language_detection_correct", "mean"),
    )
    .reset_index()
)

display(model_summary)

,pipeline_model_id,approach,n_items,n_errors,avg_latency_seconds,language_accuracy
0,deepseek-ai/DeepSeek-R1-Distill-Qwen-14B,closed_book,90,0,5.587300,1.0
1,deepseek-ai/DeepSeek-R1-Distill-Qwen-14B,rag,90,0,5.827860,1.0
2,deepseek-ai/DeepSeek-R1-Distill-Qwen-14B,retrieval_based,90,0,0.000866,1.0
3,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,closed_book,90,0,2.823803,1.0
4,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,rag,90,0,3.049872,1.0
5,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,retrieval_based,90,0,0.000901,1.0


In [ ]:
dataset_summary = (
    multi_model_results
    .groupby(["pipeline_model_id", "approach", "dataset"])
    .agg(
        n_items=("item_id", "count"),
        n_errors=("error_message", lambda x: (x != "").sum()),
        avg_latency_seconds=("latency_seconds", "mean"),
    )
    .reset_index()
)

display(dataset_summary)

,pipeline_model_id,approach,dataset,n_items,n_errors,avg_latency_seconds
0,deepseek-ai/DeepSeek-R1-Distill-Qwen-14B,closed_book,in_scope,30,0,14.599395
1,deepseek-ai/DeepSeek-R1-Distill-Qwen-14B,closed_book,out_of_scope,60,0,1.081253
2,deepseek-ai/DeepSeek-R1-Distill-Qwen-14B,rag,in_scope,30,0,15.231527
3,deepseek-ai/DeepSeek-R1-Distill-Qwen-14B,rag,out_of_scope,60,0,1.126026
4,deepseek-ai/DeepSeek-R1-Distill-Qwen-14B,retrieval_based,in_scope,30,0,0.001979
5,deepseek-ai/DeepSeek-R1-Distill-Qwen-14B,retrieval_based,out_of_scope,60,0,0.000310
6,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,closed_book,in_scope,30,0,7.369027
7,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,closed_book,out_of_scope,60,0,0.551190
8,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,rag,in_scope,30,0,7.960487
9,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B,rag,out_of_scope,60,0,0.594564


In [ ]:
display(
    multi_model_results
    .groupby(["approach", "language_detection_method", "scope_detection_method"])
    .size()
    .reset_index(name="n_rows")
)

,approach,language_detection_method,scope_detection_method,n_rows
0,closed_book,lingua,cached_qwen7b_router,118
1,closed_book,lingua,prompt_based,62
2,rag,lingua,cached_qwen7b_router,180
3,retrieval_based,lingua,cached_qwen7b_router,118
4,retrieval_based,lingua,prompt_based,62


In [ ]:
language_summary = (
    multi_model_results
    .groupby("language_detection_method")
    .agg(
        n_items=("item_id", "count"),
        language_accuracy=("language_detection_correct", "mean"),
        avg_latency_seconds=("latency_seconds", "mean"),
        n_errors=("error_message", lambda x: (x != "").sum()),
    )
    .reset_index()
)

display(language_summary)

,language_detection_method,n_items,language_accuracy,avg_latency_seconds,n_errors
0,lingua,540,1.0,2.881767,0
